### Tutorial

We will use [**Cohere Embed 4.0**](https://cohere.com/blog/embed-4) through [Qdrant Cloud Inference](https://qdrant.tech/documentation/inference/inference-api/) for generating multimodal embeddings and a [**Qdrant Collection**](qdrant.tech/documentation/manage-data/collections/) for storing and retrieving them.

> _To follow along with this example, you need a Cohere API key. Create a free one [here](https://dashboard.cohere.com/api-keys)_

In [3]:
! pip install -q qdrant-client

We will be using a [Qdrant Cloud Free Tier Cluster](/documentation/cloud/create-cluster/#free-clusters).

[Create a free cluster](https://cloud.qdrant.io/), save the associated API key and endpoint URL, and instantiate the Qdrant Client (make sure to set `cloud_inference=True` to enable Cloud Inference):

In [ ]:
from qdrant_client import QdrantClient, models
from getpass import getpass

client = QdrantClient(url=getpass("Qdrant URL: "), api_key=getpass("Qdrant API key: "), cloud_inference=True)


Let's embed a very short selection of images and their captions in the **shared embedding space**.

In [5]:
import base64

def image_to_base64_url(image_path: str) -> str:
  prefix = "data:image/png;base64"
  with open(image_path, "rb") as image_file:
    return prefix + "," + base64.b64encode(image_file.read()).decode("utf-8")

documents = [
    {"caption": "An image about plane emergency safety.", "image": "images/image-1.png"},
    {"caption": "An image about airplane components.", "image": "images/image-2.png"},
    {"caption": "An image about COVID safety restrictions.", "image": "images/image-3.png"},
    {"caption": "A confidential image about UFO sightings.", "image": "images/image-4.png"},
    {"caption": "An image about unusual footprints on Aralar 2011.", "image": "images/image-5.png"},
]

Create a **Collection**

In [6]:
COLLECTION_NAME = "multimodal-embeddings"

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "image": models.VectorParams(size=512, distance=models.Distance.COSINE),
            "text": models.VectorParams(size=512, distance=models.Distance.COSINE),
        }
    )

Now let's upload our images with captions to the **Collection**. Each image with its caption will be embedded by the Cohere model, [through Cloud Inference](https://qdrant.tech/documentation/inference/external-inference-providers/#cohere), and uploaded, as a [Point](https://qdrant.tech/documentation/concepts/points/), to the collection.

In [ ]:
from qdrant_client.context_headers import headers

cohere_api_key = getpass("Cohere API key: ")

with headers({"cohere-api-key": cohere_api_key}):
  client.upsert(
      collection_name=COLLECTION_NAME,
      points=[
          models.PointStruct(
              id=idx,
              vector={
                  "text": models.Document(
                      text=doc["caption"],
                      model="cohere/embed-v4.0",
                      options={
                          "output_dimension": 512
                      }
                  ),
                  "image": models.Image(
                      image=image_to_base64_url(doc["image"]),
                      model="cohere/embed-v4.0",
                      options={
                          "output_dimension": 512
                      }
                  ),
              },
              payload=doc
          )
          for idx, doc in enumerate(documents)
      ]
  )

Let's see what image we get for the query "*Plane components*"

In [ ]:
from PIL import Image

with headers({"cohere-api-key": cohere_api_key}):
  image_path = client.query_points(
      collection_name=COLLECTION_NAME,
      query=models.Document(
          text="Plane components",
          model="cohere/embed-v4.0",
          options={
            "output_dimension": 512
          }
      ),
      using="image",
      with_payload=["image"],
      limit=1
  ).points[0].payload['image']

Image.open(image_path)

Let's also run the same query in Italian (one of the 30+ languages supported by the model) and compare the results.

In [ ]:
with headers({"cohere-api-key": cohere_api_key}):
  image_path = client.query_points(
      collection_name=COLLECTION_NAME,
      query=models.Document(
          text="Componenti di un aereo",
          model="cohere/embed-v4.0",
          options={
            "output_dimension": 512
          }
      ),
      using="image",
      with_payload=["image"],
      limit=1
  ).points[0].payload['image']

Image.open(image_path)

Now let's do a reverse search for the following image:

In [ ]:
Image.open("images/image-2.png")

In [ ]:
with headers({"cohere-api-key": cohere_api_key}):
  client.query_points(
      collection_name=COLLECTION_NAME,
      query=models.Image(
          image=image_to_base64_url("images/image-2.png"),
          model="cohere/embed-v4.0",
          options={
            "output_dimension": 512
          }
      ),
      # Now we are searching only among text vectors with our image query
      using="text",
      with_payload=["caption"],
      limit=1
  ).points[0].payload['caption']